In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report,accuracy_score,precision_score,recall_score,f1_score)

# Load Dataset
data = pd.read_csv("pos_tags.csv", nrows=1000)
print(data.head())

# Convert into Sentences
sentences = []
temp = []

for _, row in data.iterrows():
    word = str(row["word"])
    tag = str(row["tag"])
    temp.append((word, tag))

    if len(temp) == 20:
        sentences.append(temp)
        temp = []

if len(temp) > 0:
    sentences.append(temp)

print("Total Sentences :", len(sentences))

# Train Test Split
train_data, test_data = train_test_split(sentences,test_size=0.2,random_state=42)

# Vocabulary
vocabulary = set()
tags = set()

for sentence in train_data:
    for word, tag in sentence:
        vocabulary.add(word.lower())
        tags.add(tag)

vocabulary = list(vocabulary)
tags = list(tags)

word_to_index = {
    word: i
    for i, word in enumerate(vocabulary)
}

tag_to_index = {
    tag: i
    for i, tag in enumerate(tags)
}

index_to_tag = {
    i: tag
    for tag, i in tag_to_index.items()
}

V = len(vocabulary)
T = len(tags)

print("\nVocabulary Size :", V)
print("Number of POS Tags :", T)

# HMM Matrices
initial = np.ones(T)
transition = np.ones((T, T))
emission = np.ones((T, V))

# Count Frequencies
for sentence in train_data:

    first_tag = sentence[0][1]

    initial[
        tag_to_index[first_tag]
    ] += 1

    for i, (word, tag) in enumerate(sentence):

        word = word.lower()

        tag_index = tag_to_index[tag]

        # Emission Count
        if word in word_to_index:
            emission[
                tag_index,
                word_to_index[word]
            ] += 1

        # Transition Count
        if i > 0:
            previous_tag = sentence[i - 1][1]
            transition[
                tag_to_index[previous_tag],
                tag_index
            ] += 1

# Convert to Probabilities
initial /= initial.sum()
transition /= transition.sum(
    axis=1,
    keepdims=True
)
emission /= emission.sum(
    axis=1,
    keepdims=True
)

# Log Space
log_initial = np.log(initial)
log_transition = np.log(transition)
log_emission = np.log(emission)

# Vectorized Viterbi
def viterbi(sentence):
    n = len(sentence)
    dp = np.zeros((T, n))
    backpointer = np.zeros((T, n), dtype=int)

    # First Word
    word = sentence[0].lower()
    if word in word_to_index:
        emit = log_emission[
            :,
            word_to_index[word]
        ]
    else:
        emit = np.log(np.ones(T) * 1e-10)
    dp[:, 0] = log_initial + emit

    # Remaining Words
    for i in range(1, n):
        word = sentence[i].lower()
        if word in word_to_index:
            emit = log_emission[
                :,
                word_to_index[word]
            ]
        else:
            emit = np.log(np.ones(T) * 1e-10)
        scores = dp[:, i - 1][:, None] + log_transition
        backpointer[:, i] = np.argmax(
            scores,
            axis=0
        )
        dp[:, i] = np.max(
            scores,
            axis=0
        ) + emit

    # Backtracking
    best = np.argmax(dp[:, -1])
    result = [best]
    for i in range(n - 1, 0, -1):
        best = backpointer[best, i]
        result.append(best)
    result.reverse()
    return [
        index_to_tag[i]
        for i in result
    ]

# Single Prediction
sentence = "Artificial Intelligence improves healthcare systems".split()
prediction = viterbi(sentence)

print("\nPrediction")
print("----------------")

for word, tag in zip(sentence, prediction):
    print(word, "----->", tag)

# Evaluation
actual = []
predicted = []

for sentence in test_data:
    words = [
        w
        for w, t in sentence
    ]

    true = [
        t
        for w, t in sentence
    ]

    pred = viterbi(words)
    actual.extend(true)
    predicted.extend(pred)

print("\nAccuracy")
print(accuracy_score(actual, predicted))

print("\nPrecision")
print(
    precision_score(
        actual,
        predicted,
        average="weighted",
        zero_division=0
    )
)

print("\nRecall")
print(
    recall_score(
        actual,
        predicted,
        average="weighted",
        zero_division=0
    )
)

print("\nF1 Score")
print(
    f1_score(
        actual,
        predicted,
        average="weighted",
        zero_division=0
    )
)

print("\nClassification Report")

print(
    classification_report(
        actual,
        predicted,
        zero_division=0
    )
)

# Test on Unseen Sentences
test_sentences = [
    "The cat sat on the mat",
    "Machine learning changes the world",
    "Students study artificial intelligence",
    "OpenAI develops powerful AI models",
    "Birds fly in the sky"
]

print("\nUnseen Sentence Predictions")
print("--------------------------------")

for s in test_sentences:
    words = s.split()
    tags = viterbi(words)
    print("\nSentence :", s)

    for w, t in zip(words, tags):
        print(w, "----->", t)


   sentence_id    word  tag
0            0      aa   NN
1            1     aaa   NN
2            2     aah   NN
3            3   aahed  VBN
4            4  aahing  VBG
Total Sentences : 50

Vocabulary Size : 800
Number of POS Tags : 9

Prediction
----------------
Artificial -----> NN
Intelligence -----> NN
improves -----> NN
healthcare -----> NN
systems -----> NN

Accuracy
0.58

Precision
0.33640000000000003

Recall
0.58

F1 Score
0.4258227848101266

Classification Report
              precision    recall  f1-score   support

          JJ       0.00      0.00      0.00        25
          NN       0.58      1.00      0.73       116
         NNS       0.00      0.00      0.00        29
          RB       0.00      0.00      0.00         5
         VBD       0.00      0.00      0.00         2
         VBG       0.00      0.00      0.00        12
         VBN       0.00      0.00      0.00        11

    accuracy                           0.58       200
   macro avg       0.08      0.14  